# 01.3 Autograd

`Autograd` is one of the core capabilities of `PyTorch`.

If you really understand this notebook, `loss.backward()` will stop being something you merely copy.

Key concepts:

- `requires_grad`
- `backward()`
- gradient accumulation
- `no_grad()` and `detach()`

## Learning Goals

After this notebook, you should be able to:

1. Understand what `requires_grad=True` means.
2. Read a simple computational graph.
3. Use `backward()` to compute gradients.
4. Understand why gradients accumulate.
5. Use `torch.no_grad()` correctly.
6. Understand what `detach()` does.

In [ ]:
import torch

## `requires_grad` and the Computational Graph

When a tensor has `requires_grad=True`, `PyTorch` starts tracking operations involving it.

These connected operations form the computational graph.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

print("x =", x)
print("y =", y)
print("x.requires_grad =", x.requires_grad)
print("y.requires_grad =", y.requires_grad)
print("y.grad_fn =", y.grad_fn)

A good first intuition is:

- `x` is an input we want gradients for
- `y` is produced from `x` through a sequence of operations
- `grad_fn` indicates tracked computation history

## Using `backward()` to Compute Gradients

If the output is a scalar, you can call `backward()` directly.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1
y.backward()

print("x.grad =", x.grad)

Manual check:

- `y = x^2 + 3x + 1`
- `dy/dx = 2x + 3`

So `x.grad == 7`.


In [ ]:
# Exercise 1
# Let y = 4x^2 - x
#1. Set x=3 with requires_grad=True
# 2. Use backward() to compute the gradient
# 3. Print x.grad

# x =
# y =
# y.backward()
# print(x.grad)

In [ ]:
# Exercise 1 Reference Solution

x = torch.tensor(3.0, requires_grad=True)
y = 4 * x ** 2 - x
y.backward()
print(x.grad)

# manually: dy/dx = 8x - 1, at x=3 => 23

## Non-Scalar Outputs

If the output is not a scalar, `backward()` needs an additional gradient argument.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2

y.backward(gradient=torch.ones_like(y))
print("x.grad =", x.grad)

Passing `torch.ones_like(y)` is equivalent to weighting each output term equally.

In practice, the training `loss` is usually already a scalar, so the common pattern is simply:

- `loss.backward()`

## Gradient Accumulation

This is a very important, very common, and very easy-to-miss point.

In `PyTorch`, gradients accumulate in `.grad` by default.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("after first backward / after first backward:", x.grad)

y2 = 3 * x
y2.backward()
print("after second backward / after second backward:", x.grad)

Why does this happen?

- First:`d(x^2)/dx = 2x = 4`
- Second:`d(3x)/dx = 3`
- Accumulated result:`4 + 3 = 7`

This is one reason why training loops often call `optimizer.zero_grad()`.


In [ ]:
# Exercise 2
# Reproduce the experiment:
# 1. x=1, requires_grad=True
# 3. Then call backward() on z=2x
# 4. Observe the accumulated x.grad result

# x =
# y =
# y.backward()
# z =
# z.backward()
# print(x.grad)

In [ ]:
# Exercise 2 Reference Solution

x = torch.tensor(1.0, requires_grad=True)
y = x ** 3
y.backward()
z = 2 * x
z.backward()
print(x.grad)

# dy/dx = 3, dz/dx = 2, total = 5

## Manually Zeroing Gradients

Before updating parameters, you usually clear old gradients first.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
print("before zeroing / before zeroing:", x.grad)

x.grad.zero_()
print("after zeroing / after zeroing:", x.grad)

z = 3 * x
z.backward()
print("after backward again / after backward again:", x.grad)

## 6. `torch.no_grad()`

Sometimes you do not want `PyTorch` to track gradients, for example:

- inference
- validation
- plain numeric inspection

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x * 5

print("y =", y)
print("y.requires_grad =", y.requires_grad)

Main value of `no_grad()`:

- saves graph overhead and memory
- avoids unnecessary gradient tracking

## 7. `detach()`

`detach()` returns a new tensor view that shares data but no longer participates in the current graph.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = 2 * x
z = y.detach()

print("y.requires_grad =", y.requires_grad)
print("z.requires_grad =", z.requires_grad)

y_sum = y.sum()
y_sum.backward()
print("x.grad =", x.grad)

Intuitively, you can think of it as:

- `y` is still connected to the graph
- `z` has been cut away from the graph

In [ ]:
# Exercise 3
# Decide which tensors below will track gradients.
#
# x = torch.tensor(2.0, requires_grad=True)
# a = x * 2
# with torch.no_grad():
#     b = x * 3
# c = a.detach()
#
# print(a.requires_grad)
# print(b.requires_grad)
# print(c.requires_grad)

In [ ]:
# Exercise 3 Reference Solution

x = torch.tensor(2.0, requires_grad=True)
a = x * 2
with torch.no_grad():
    b = x * 3
c = a.detach()

print(a.requires_grad)
print(b.requires_grad)
print(c.requires_grad)

## A Tiny Training-Flavored Example

The example below is not yet a full training loop, but it is already very close.


In [ ]:
w = torch.tensor(0.5, requires_grad=True)
x = torch.tensor(2.0)
target = torch.tensor(4.0)

pred = w * x
loss = (pred - target) ** 2
loss.backward()

print("pred =", pred.item())
print("loss =", loss.item())
print("w.grad =", w.grad.item())

The core training chain already appears here:

- parameter: `w`
- forward pass: `pred = w * x`
- loss: `(pred - target)^2`
- backward pass: `loss.backward()`

When you learn the training loop later, this chain will become the full version.


## Summary

You should now be able to answer:

1. What exactly does `requires_grad=True` mean?
2. Why can scalar outputs call `backward()` directly?
3. Why does `.grad` accumulate?
4. What is the difference between `no_grad()` and `detach()`?
5. Why do we usually clear gradients before training steps?

Suggested next step:

- `DataLoader`, or directly into `nn.Module` and the training loop.